In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import numpy as np

In [ ]:
import pandas as pd

Features_valid = pd.read_csv(
    "/content/drive/MyDrive/Features_valid.csv"
)
Features = pd.read_csv(
    "/content/drive/MyDrive/Features.csv"
)
display(Features_valid, Features)

,run number,VB,vibs_mean,vibs_std,AEs_mean,AEs_std
0,1,0.00,0.648509,0.078750,0.209068,0.037541
1,4,0.11,0.586600,0.060225,0.227271,0.038139
2,6,0.20,0.524458,0.036511,0.237914,0.030290
3,7,0.24,0.443222,0.030803,0.333051,0.049276
4,8,0.29,0.442355,0.032362,0.292756,0.036115
5,9,0.28,0.441375,0.027858,0.384274,0.054352
6,10,0.29,0.407035,0.025583,0.339209,0.039691
7,11,0.38,0.383948,0.023873,0.425877,0.051597
8,12,0.40,0.481121,0.035500,0.387235,0.047492
9,13,0.43,0.393711,0.021022,0.387914,0.049563


,run number,VB,vibs_mean,vibs_std,AEs_mean,AEs_std
0,1,0.00,0.648509,0.078750,0.209068,0.037541
1,2,NaN,0.648881,0.078405,0.205762,0.035949
2,3,NaN,0.659785,0.087694,0.227077,0.045176
3,4,0.11,0.586600,0.060225,0.227271,0.038139
4,5,NaN,0.551760,0.037256,0.250885,0.032274
5,6,0.20,0.524458,0.036511,0.237914,0.030290
6,7,0.24,0.443222,0.030803,0.333051,0.049276
7,8,0.29,0.442355,0.032362,0.292756,0.036115
8,9,0.28,0.441375,0.027858,0.384274,0.054352
9,10,0.29,0.407035,0.025583,0.339209,0.039691


In [ ]:
from mill_function import fit_one, fit_two

In [ ]:
a_vib, b_vib, result_vib = fit_one(Features_valid, "vibs_mean")

display(result_vib)

from mill_function import fit_one, fit_two

a_AE, b_AE, result_AE = fit_one(Features_valid, "AEs_mean")

display(result_AE)

,run number,VB,estimated VB,VB residual
0,1,0.00,0.013861,-0.013861
1,4,0.11,0.110936,-0.000936
2,6,0.20,0.208376,-0.008376
3,7,0.24,0.335755,-0.095755
4,8,0.29,0.337116,-0.047116
5,9,0.28,0.338652,-0.058652
6,10,0.29,0.392497,-0.102497
7,11,0.38,0.428699,-0.048699
8,12,0.40,0.276329,0.123671
9,13,0.43,0.413390,0.016610


,run number,VB,estimated VB,VB residual
0,1,0.00,0.097389,-0.097389
1,4,0.11,0.123782,-0.013782
2,6,0.20,0.139213,0.060787
3,7,0.24,0.277153,-0.037153
4,8,0.29,0.218729,0.071271
5,9,0.28,0.351420,-0.071420
6,10,0.29,0.286081,0.003919
7,11,0.38,0.411740,-0.031740
8,12,0.40,0.355714,0.044286
9,13,0.43,0.356698,0.073302


In [ ]:
a_two, b_two, c_two, result_two = fit_two(
    Features_valid, "vibs_mean", "AEs_mean"
)

display(result_two)

,run number,VB,estimated VB,VB residual
0,1,0.00,0.042135,-0.042135
1,4,0.11,0.100580,0.009420
2,6,0.20,0.151734,0.048266
3,7,0.24,0.298590,-0.058590
4,8,0.29,0.259477,0.030523
5,9,0.28,0.350240,-0.070240
6,10,0.29,0.328338,-0.038338
7,11,0.38,0.428795,-0.048795
8,12,0.40,0.327142,0.072858
9,13,0.43,0.385022,0.044978


In [ ]:
Compare = pd.DataFrame({
    "model":["vibs","AEs","Both"],
    "MAE":[np.mean(abs(result_vib["VB residual"])),np.mean(abs(result_AE["VB residual"])),np.mean(abs(result_two["VB residual"]))]
})
display(Compare)

,model,MAE
0,vibs,0.057829
1,AEs,0.051077
2,Both,0.046308


**Already prove the two-feature estimate is better than one-feature when use all runs data.**

**Now leave some runs out to test the model**

*test last 4 runs:13,14,15,17*

In [ ]:
run_reorder = Features_valid.sort_values("run number")
train = run_reorder.iloc[:-4]
test = run_reorder.iloc[-4:]


In [ ]:
# two feature test
a_test, b_test, c_test, two_train = fit_two(train,"vibs_mean","AEs_mean")
VB_test = a_test * test["vibs_mean"] + b_test * test["AEs_mean"] + c_test
Two_VB_test_residual = test["VB"] - VB_test
MAE_test = np.mean(abs(Two_VB_test_residual))

Test_table = pd.DataFrame({
    "test VB":test["VB"],
    "VB_est":VB_test,
    "residual":Two_VB_test_residual

})
display(Test_table)
print("MAE=",MAE_test)

,test VB,VB_est,residual
9,0.43,0.362012,0.067988
10,0.45,0.394885,0.055115
11,0.50,0.400595,0.099405
12,0.44,0.445438,-0.005438


MAE= 0.05698616767459978


In [ ]:
# Vibration: fit on train, predict test
a_vibs_test, b_vibs_test, _= fit_one(train, "vibs_mean")
VB_vibs_test = a_vibs_test * test["vibs_mean"] + b_vibs_test
residual_vibs_test = test["VB"] - VB_vibs_test
MAE_vibs_test = np.mean(abs(residual_vibs_test))

# AE: fit on train, predict test
a_AEs_test, b_AEs_test, _= fit_one(train, "AEs_mean")
VB_AEs_test = a_AEs_test * test["AEs_mean"] + b_AEs_test
residual_AEs_test = test["VB"] - VB_AEs_test
MAE_AEs_test = np.mean(abs(residual_AEs_test))

display(pd.DataFrame({
    "Model": ["Vibration", "AE", "Both"],
    "Test MAE": [MAE_vibs_test, MAE_AEs_test, MAE_test]
}))

,Model,Test MAE
0,Vibration,0.114171
1,AE,0.062440
2,Both,0.056986


**Compare the residual run by run to find to find where the improvement comes from.**

**Contributed by all runs? Mainly from one run?**

In [ ]:
print(test
    )

    run number    VB  vibs_mean  vibs_std  AEs_mean   AEs_std
9           13  0.43   0.393711  0.021022  0.387914  0.049563
10          14  0.45   0.404036  0.022233  0.434103  0.052908
11          15  0.50   0.417644  0.023860  0.450814  0.069436
12          17  0.44   0.414159  0.035573  0.500901  0.076089


In [ ]:
Test_improve = pd.DataFrame({
    "run number": test["run number"],
    "VB": test["VB"],
    "vib-Residual": residual_vibs_test,
    "AE-Residual":residual_AEs_test,
    "Two-Residual":Two_VB_test_residual
})
display(Test_improve)

,run number,VB,vib-Residual,AE-Residual,Two-Residual
9,13,0.43,0.071832,0.082077,0.067988
10,14,0.45,0.104921,0.035656,0.055115
11,15,0.50,0.172174,0.061625,0.099405
12,17,0.44,0.107755,-0.070402,-0.005438


**The two-feature model has a lower MAE on this test set, but its advantage over AE-only depends strongly on the last test run**

**Now use last 5 runs to test to check model consistency (*resuable function test_run*)**

In [ ]:
from mill_function import test_run

In [ ]:
Test_improve, MAE_compare = test_run(Features_valid, 12)
display(Test_improve, MAE_compare)

,run number,VB,vib-Residual,AE-Residual,Two-Residual
12,17,0.44,0.065551,-0.113317,-0.073356


,Model,Test MAE
0,Vibration,0.065551
1,AE,0.113317
2,Both,0.073356


**start from 4 is because we have 3 unknown varible in two-feature regression**

In [ ]:
two_better = 0
two_worse = 0
for i in range(4,len(Features_valid)):
  Test_improve, MAE_compare = test_run(Features_valid, i)

  vib_mae = MAE_compare["Test MAE"].iloc[0]
  ae_mae = MAE_compare["Test MAE"].iloc[1]
  two_mae = MAE_compare["Test MAE"].iloc[2]

  if two_mae < vib_mae and two_mae < ae_mae:
      two_better += 1
  else:
      two_worse += 1

print("Both-better:", two_better)
print("Both-worth:", two_worse)




Both-better: 3
Both-worth: 6


**The two-feature model outperforms both single-feature models only when trained on 9, 10, or 11 runs**

**In the previous test method, increasing the number of training runs also changed the size and composition of the test set. Therefore, changes in test MAE could reflect both changes in the fitted model and changes in the runs being evaluated.**

**the following test method will contain two parts: 1.consistently test the last 3 run-11,12,13 and gradually increase the number of training run from 4, to test whether adding training data improves prediction on the same test runs. The findings may depend on this particular test set.**

**2.gradually increase the number of training from 4 and consistenly test the next 3 runs, to test how consistently each model predicts the next few runs as machining progresses. Changes in MAE reflect changes in both the training data and the test runs.**